In [6]:
import sys

import numpy as np
import sklearn
import torch

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("PyTorch:", torch.__version__)

Python: 3.13.18
NumPy: 2.4.6
scikit-learn: 1.6.1
PyTorch: 2.11.0+cpu


In [7]:
from sklearn.datasets import load_digits

X, y = load_digits(return_X_y=True)

print("Размер X:", X.shape)
print("Размер y:", y.shape)
print("Минимальное значение пикселя:", X.min())
print("Максимальное значение пикселя:", X.max())
print("Классы:", np.unique(y))

Размер X: (1797, 64)
Размер y: (1797,)
Минимальное значение пикселя: 0.0
Максимальное значение пикселя: 16.0
Классы: [0 1 2 3 4 5 6 7 8 9]


In [ ]:
from sklearn.model_selection import train_test_split

split_seed = 0
all_indices = np.arange(len(y))

rest_indices, test_indices = train_test_split(
    all_indices,
    test_size=0.2,
    stratify=y,
    random_state=split_seed,
)

train_indices, val_indices = train_test_split(
    rest_indices,
    test_size=0.25,
    stratify=y[rest_indices],
    random_state=split_seed,
)

print("train:", len(train_indices))
print("val:", len(val_indices))
print("test:", len(test_indices))

print("Пересечение train и val:", len(set(train_indices) & set(val_indices)))
print("Пересечение train и test:", len(set(train_indices) & set(test_indices)))
print("Пересечение val и test:", len(set(val_indices) & set(test_indices)))

combined = set(train_indices) | set(val_indices) | set(test_indices)
print("Всего уникальных индексов:", len(combined))

for name, indices in [
    ("train", train_indices),
    ("val", val_indices),
    ("test", test_indices),
]:
    print(name, "классы:", np.unique(y[indices]))

train: 1077
val: 360
test: 360
Пересечение train и val: 0
Пересечение train и test: 0
Пересечение val и test: 0
Всего уникальных индексов: 1797
train классы: [0 1 2 3 4 5 6 7 8 9]
val классы: [0 1 2 3 4 5 6 7 8 9]
test классы: [0 1 2 3 4 5 6 7 8 9]


In [ ]:
from pathlib import Path
import hashlib
import json

data_directory = Path("data")
data_directory.mkdir(exist_ok=True)

split_data = {
    "dataset": "sklearn.datasets.load_digits",
    "split_seed": split_seed,
    "train": sorted(int(index) for index in train_indices),
    "val": sorted(int(index) for index in val_indices),
    "test": sorted(int(index) for index in test_indices),
}

split_path = data_directory / "split.json"

with split_path.open("w", encoding="utf-8", newline="\n") as file:
    json.dump(split_data, file, ensure_ascii=False, indent=2, sort_keys=True)
    file.write("\n")

split_sha256 = hashlib.sha256(split_path.read_bytes()).hexdigest()

print("Создан файл:", split_path)
print("Размер файла:", split_path.stat().st_size, "байт")
print("SHA-256:", split_sha256)

Создан файл: data/split.json
Размер файла: 16975 байт
SHA-256: d3ba070d3282c94e67fcf0f0bd7c10e0efc97dbcb00c07397313e14d38b5931a


In [ ]:
manifest_data = {
    "schema_version": 1,
    "datasets": {
        "load_digits": {
            "source": "sklearn.datasets.load_digits",
            "package": "scikit-learn",
            "package_version": sklearn.__version__,
            "total_records": int(len(y)),
            "features_per_record": int(X.shape[1]),
            "classes": [int(value) for value in np.unique(y)],
            "feature_value_range": [
                float(X.min()),
                float(X.max()),
            ],
            "selection": {
                "included_records": int(len(y)),
                "excluded_records": 0,
                "rule": "Использованы все записи набора load_digits без исключений",
            },
            "split": {
                "file": "data/split.json",
                "sha256": split_sha256,
                "split_seed": split_seed,
                "rule": (
                    "Стратифицированное разбиение: сначала 20% в test, "
                    "затем 25% остатка в val"
                ),
                "sizes": {
                    "train": int(len(train_indices)),
                    "val": int(len(val_indices)),
                    "test": int(len(test_indices)),
                },
            },
        }
    },
}

manifest_path = data_directory / "manifest.json"

with manifest_path.open("w", encoding="utf-8", newline="\n") as file:
    json.dump(manifest_data, file, ensure_ascii=False, indent=2, sort_keys=True)
    file.write("\n")

print("Создан файл:", manifest_path)
print("Размер файла:", manifest_path.stat().st_size, "байт")
print("Версия scikit-learn:", sklearn.__version__)
print("Хеш split.json в манифесте:", manifest_data["datasets"]["load_digits"]["split"]["sha256"])

Создан файл: data/manifest.json
Размер файла: 1075 байт
Версия scikit-learn: 1.6.1
Хеш split.json в манифесте: d3ba070d3282c94e67fcf0f0bd7c10e0efc97dbcb00c07397313e14d38b5931a


In [ ]:
from pathlib import Path

print("Текущая папка:", Path.cwd())
print("Папка data существует:", Path("data").exists())

if Path("data").exists():
    print("Файлы внутри data:", [path.name for path in Path("data").iterdir()])

Текущая папка: /content
Папка data существует: True
Файлы внутри data: ['manifest.json', 'split.json']


In [1]:
%pip install -q mlflow==3.18.0 optuna==5.0.0 PyYAML==6.0.3

Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 10, in <module>
    sys.exit(main())
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/main.py", line 47, in main
    from pip._internal.cli.autocompletion import autocomplete
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/autocompletion.py", line 12, in <module>
    from pip._internal.cli.main_parser import create_main_parser
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/main_parser.py", line 11, in <module>
    from pip._internal.cli import cmdoptions
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/cmdoptions.py", line 28, in <module>
    from pip._internal.cli.parser import ConfigOptionParser
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/parser.py", line 20, in <module>
    from pip._internal.configuration import Configuration, ConfigurationError
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/configuration.p

In [2]:
from pathlib import Path
import os
import zipfile

zip_files = sorted(
    Path("/content").glob("lr0-experiment-harness*.zip"),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

if not zip_files:
    raise FileNotFoundError("ZIP-файл репозитория не найден")

archive_path = zip_files[0]
extract_directory = Path("/content/lr0_project")
extract_directory.mkdir(exist_ok=True)

with zipfile.ZipFile(archive_path) as archive:
    archive.extractall(extract_directory)

project_directories = [
    path
    for path in extract_directory.iterdir()
    if path.is_dir()
]

if not project_directories:
    raise FileNotFoundError("Папка проекта после распаковки не найдена")

project_directory = project_directories[0]
os.chdir(project_directory)

print("Рабочая папка:", Path.cwd())

for required_path in [
    "prepare_data.py",
    "train.py",
    "configs/base.yaml",
    "data/split.json",
    "data/manifest.json",
]:
    print(required_path, "—", Path(required_path).exists())

Рабочая папка: /content/lr0_project/lr0-experiment-harness-main
prepare_data.py — True
train.py — True
configs/base.yaml — True
data/split.json — True
data/manifest.json — True


In [3]:
%run prepare_data.py

Подготовка данных завершена
train: 1077
val: 360
test: 360
SHA-256 split.json: d3ba070d3282c94e67fcf0f0bd7c10e0efc97dbcb00c07397313e14d38b5931a


In [8]:
%pip install mlflow==3.18.0 optuna==5.0.0 PyYAML==6.0.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 87.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 91.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 77.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 51.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19/19 [mlflow]


In [9]:
import mlflow
import optuna
import yaml

print("MLflow:", mlflow.__version__)
print("Optuna:", optuna.__version__)
print("PyYAML:", yaml.__version__)

MLflow: 3.18.0
Optuna: 5.0.0
PyYAML: 6.0.3


In [10]:
%run train.py --config configs/base.yaml --series debug

2026/11/06 00:54:25 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/06 00:54:25 INFO mlflow.store.db.utils: Updating database tables
2026/11/06 00:54:29 INFO mlflow.tracking.fluent: Experiment with name 'lr0' does not exist. Creating a new experiment.


Запуск завершён: 20261106T005422Z_9f46c007
Каталог: runs/20261106T005422Z_9f46c007
Метрики: {'val_loss': 0.19422872364521027, 'training_time_seconds': 0.974481108999953, 'val_acc': 0.95}


In [11]:
%run train.py --config configs/base.yaml --series repeatability_check --seed 0
%run train.py --config configs/base.yaml --series repeatability_check --seed 1

Запуск завершён: 20261106T005559Z_364e3d53
Каталог: runs/20261106T005559Z_364e3d53
Метрики: {'val_loss': 0.19422872364521027, 'training_time_seconds': 0.8430049550004242, 'val_acc': 0.95}
Запуск завершён: 20261106T005601Z_b6c5eef4
Каталог: runs/20261106T005601Z_b6c5eef4
Метрики: {'val_loss': 0.1885450929403305, 'training_time_seconds': 0.8145036129994878, 'val_acc': 0.9611111111111111}


In [18]:
import csv
import json
from pathlib import Path

import numpy as np


def read_csv(path):
    with path.open("r", encoding="utf-8", newline="") as file:
        return list(csv.DictReader(file))


completed_runs = []

for run_directory in Path("runs").iterdir():
    metadata_path = run_directory / "metadata.json"

    if not metadata_path.exists():
        continue

    metadata = json.loads(
        metadata_path.read_text(encoding="utf-8")
    )

    if metadata["status"] == "completed":
        completed_runs.append((run_directory, metadata))

seed_zero_runs = sorted(
    [
        item
        for item in completed_runs
        if item[1]["seed"] == 0
    ],
    key=lambda item: item[1]["started_at"],
)

seed_one_runs = sorted(
    [
        item
        for item in completed_runs
        if item[1]["seed"] == 1
    ],
    key=lambda item: item[1]["started_at"],
)

if len(seed_zero_runs) < 2:
    raise ValueError("Нужно два завершённых запуска с seed=0")

if not seed_one_runs:
    raise ValueError("Нужен один завершённый запуск с seed=1")

seed0_first = seed_zero_runs[-2][0]
seed0_second = seed_zero_runs[-1][0]
seed1_run = seed_one_runs[-1][0]

predictions_0_first = read_csv(
    seed0_first / "predictions.csv"
)
predictions_0_second = read_csv(
    seed0_second / "predictions.csv"
)
predictions_1 = read_csv(
    seed1_run / "predictions.csv"
)

history_0_first = read_csv(seed0_first / "history.csv")
history_0_second = read_csv(seed0_second / "history.csv")

deterministic_columns = [
    "epoch",
    "train_loss",
    "val_loss",
    "val_acc",
]

history_0_first_deterministic = [
    {
        column: row[column]
        for column in deterministic_columns
    }
    for row in history_0_first
]

history_0_second_deterministic = [
    {
        column: row[column]
        for column in deterministic_columns
    }
    for row in history_0_second
]

same_seed_predictions_match = (
    predictions_0_first == predictions_0_second
)

same_seed_history_matches = (
    history_0_first_deterministic
    == history_0_second_deterministic
)

different_seed_predictions_differ = (
    predictions_0_second != predictions_1
)

metrics = json.loads(
    (seed0_second / "metrics.json").read_text(
        encoding="utf-8"
    )
)

recomputed_accuracy = np.mean(
    [
        row["y_true"] == row["y_pred"]
        for row in predictions_0_second
        if row["split"] == "val"
    ]
)

simple_accuracy = np.mean(
    np.array([1, 2, 3, 3])
    == np.array([1, 2, 3, 4])
)

print(
    "Одинаковые seed — predictions совпали:",
    same_seed_predictions_match,
)
print(
    "Одинаковые seed — история метрик совпала:",
    same_seed_history_matches,
)
print(
    "Разные seed — predictions различаются:",
    different_seed_predictions_differ,
)
print(
    "Точность из predictions.csv:",
    recomputed_accuracy,
)
print(
    "Точность из metrics.json:",
    metrics["val_acc"],
)
print(
    "Проверочный пример даёт:",
    simple_accuracy,
)

assert same_seed_predictions_match
assert same_seed_history_matches
assert different_seed_predictions_differ
assert recomputed_accuracy == metrics["val_acc"]
assert simple_accuracy == 0.75

print("Все проверки пройдены")

Одинаковые seed — predictions совпали: True
Одинаковые seed — история метрик совпала: True
Разные seed — predictions различаются: True
Точность из predictions.csv: 0.95
Точность из metrics.json: 0.95
Проверочный пример даёт: 0.75
Все проверки пройдены


In [13]:
from pathlib import Path
import os
import subprocess

base_directory = Path("/content/lr0_git")
clone_directory = base_directory
counter = 1

while clone_directory.exists():
    clone_directory = Path(f"/content/lr0_git_{counter}")
    counter += 1

subprocess.run(
    [
        "git",
        "clone",
        "https://github.com/marchukmarina-creator/lr0-experiment-harness.git",
        str(clone_directory),
    ],
    check=True,
)

os.chdir(clone_directory)

commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

git_status = subprocess.run(
    ["git", "status", "--porcelain"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

print("Рабочая папка:", Path.cwd())
print("Коммит:", commit)
print("Незакоммиченные изменения:", repr(git_status))

Рабочая папка: /content/lr0_git
Коммит: a90be95147739f93924ccc94f57c489b8a4ce78c
Незакоммиченные изменения: ''


In [14]:
import subprocess
import sys

planned_runs = [
    ("repeatability", 0),
    ("repeatability", 0),
    ("base_seeds", 1),
    ("base_seeds", 2),
    ("base_seeds", 3),
    ("base_seeds", 4),
    ("base_seeds", 5),
]

for number, (series, seed) in enumerate(
    planned_runs,
    start=1,
):
    print(
        f"\nЗапуск {number}/{len(planned_runs)}: "
        f"series={series}, seed={seed}"
    )

    completed = subprocess.run(
        [
            sys.executable,
            "train.py",
            "--config",
            "configs/base.yaml",
            "--series",
            series,
            "--seed",
            str(seed),
            "--final-run",
        ],
        check=True,
        capture_output=True,
        text=True,
    )

    print(completed.stdout)


Запуск 1/7: series=repeatability, seed=0
Запуск завершён: 20261106T010203Z_8dc86cef
Каталог: runs/20261106T010203Z_8dc86cef
Метрики: {'val_loss': 0.19422872364521027, 'training_time_seconds': 1.1292691769997418, 'val_acc': 0.95}


Запуск 2/7: series=repeatability, seed=0


CalledProcessError: Command '['/usr/bin/python3', 'train.py', '--config', 'configs/base.yaml', '--series', 'repeatability', '--seed', '0', '--final-run']' returned non-zero exit status 1.

In [17]:
remaining_runs = [
    ("repeatability", 0),
    ("base_seeds", 1),
    ("base_seeds", 2),
    ("base_seeds", 3),
    ("base_seeds", 4),
    ("base_seeds", 5),
]

for number, (series, seed) in enumerate(
    remaining_runs,
    start=2,
):
    print(
        f"\nЗапуск {number}/7: "
        f"series={series}, seed={seed}"
    )

    completed = subprocess.run(
        [
            sys.executable,
            "train.py",
            "--config",
            "configs/base.yaml",
            "--series",
            series,
            "--seed",
            str(seed),
            "--final-run",
        ],
        check=True,
        capture_output=True,
        text=True,
    )

    print(completed.stdout)


Запуск 2/7: series=repeatability, seed=0
Запуск завершён: 20261106T010704Z_50a667a4
Каталог: runs/20261106T010704Z_50a667a4
Метрики: {'val_loss': 0.19422872364521027, 'training_time_seconds': 0.8169729470000675, 'val_acc': 0.95}


Запуск 3/7: series=base_seeds, seed=1
Запуск завершён: 20261106T010716Z_5c9f064f
Каталог: runs/20261106T010716Z_5c9f064f
Метрики: {'val_loss': 0.1885450929403305, 'training_time_seconds': 0.7927273090001563, 'val_acc': 0.9611111111111111}


Запуск 4/7: series=base_seeds, seed=2
Запуск завершён: 20261106T010726Z_1fe3536a
Каталог: runs/20261106T010726Z_1fe3536a
Метрики: {'val_loss': 0.20081312954425812, 'training_time_seconds': 0.991680838000093, 'val_acc': 0.9583333333333334}


Запуск 5/7: series=base_seeds, seed=3
Запуск завершён: 20261106T010737Z_0cac5df1
Каталог: runs/20261106T010737Z_0cac5df1
Метрики: {'val_loss': 0.18337346613407135, 'training_time_seconds': 1.022417562000328, 'val_acc': 0.9555555555555556}


Запуск 6/7: series=base_seeds, seed=4
Запуск 

In [19]:
import json
import statistics
from pathlib import Path

import pandas as pd


records = []

for run_directory in Path("runs").iterdir():
    metadata_path = run_directory / "metadata.json"
    metrics_path = run_directory / "metrics.json"

    if not metadata_path.exists() or not metrics_path.exists():
        continue

    metadata = json.loads(
        metadata_path.read_text(encoding="utf-8")
    )
    metrics = json.loads(
        metrics_path.read_text(encoding="utf-8")
    )

    if (
        metadata["status"] == "completed"
        and metadata["series"] in {
            "repeatability",
            "base_seeds",
        }
    ):
        records.append(
            {
                "run_id": metadata["run_id"],
                "series": metadata["series"],
                "seed": metadata["seed"],
                "val_acc": metrics["val_acc"],
                "time_seconds": metrics[
                    "training_time_seconds"
                ],
                "directory": run_directory,
            }
        )

base_runs = sorted(
    [
        record
        for record in records
        if record["series"] == "base_seeds"
    ],
    key=lambda record: record["seed"],
)

if len(base_runs) != 5:
    raise ValueError(
        f"Ожидалось 5 запусков base_seeds, найдено: "
        f"{len(base_runs)}"
    )

print("Результаты base_seeds:")

for record in base_runs:
    print(
        f"seed={record['seed']}: "
        f"val_acc={record['val_acc']:.4f}, "
        f"time={record['time_seconds']:.4f} с"
    )

accuracies = [
    record["val_acc"]
    for record in base_runs
]

print("\nСводка точности:")
print("Минимум:", min(accuracies))
print("Медиана:", statistics.median(accuracies))
print("Максимум:", max(accuracies))
print("Размах:", max(accuracies) - min(accuracies))

seed1 = base_runs[0]
seed2 = base_runs[1]

predictions_1 = pd.read_csv(
    seed1["directory"] / "predictions.csv"
)
predictions_2 = pd.read_csv(
    seed2["directory"] / "predictions.csv"
)

comparison = predictions_1.merge(
    predictions_2,
    on=["split", "index", "y_true"],
    suffixes=("_seed1", "_seed2"),
)

seed1_correct = (
    comparison["y_pred_seed1"] == comparison["y_true"]
)
seed2_correct = (
    comparison["y_pred_seed2"] == comparison["y_true"]
)

print("\nПопарное сравнение seed 1 и seed 2:")
print(
    "seed 1 верно, seed 2 ошибся:",
    int((seed1_correct & ~seed2_correct).sum()),
)
print(
    "seed 2 верно, seed 1 ошибся:",
    int((seed2_correct & ~seed1_correct).sum()),
)

repeatability_runs = sorted(
    [
        record
        for record in records
        if record["series"] == "repeatability"
    ],
    key=lambda record: record["run_id"],
)

first_predictions = pd.read_csv(
    repeatability_runs[-2]["directory"]
    / "predictions.csv"
)
second_predictions = pd.read_csv(
    repeatability_runs[-1]["directory"]
    / "predictions.csv"
)

print(
    "\nПовторяемость seed=0:",
    first_predictions.equals(second_predictions),
)

Результаты base_seeds:
seed=1: val_acc=0.9611, time=0.7927 с
seed=2: val_acc=0.9583, time=0.9917 с
seed=3: val_acc=0.9556, time=1.0224 с
seed=4: val_acc=0.9556, time=0.8126 с
seed=5: val_acc=0.9583, time=0.8247 с

Сводка точности:
Минимум: 0.9555555555555556
Медиана: 0.9583333333333334
Максимум: 0.9611111111111111
Размах: 0.005555555555555536

Попарное сравнение seed 1 и seed 2:
seed 1 верно, seed 2 ошибся: 1
seed 2 верно, seed 1 ошибся: 0

Повторяемость seed=0: True


In [24]:
search_result = subprocess.run(
    [
        sys.executable,
        "search.py",
        "--target-trials",
        "40",
    ],
    capture_output=True,
    text=True,
)

print(search_result.stdout)

if search_result.stderr:
    print("Сообщения Optuna:")
    print(search_result.stderr)

if search_result.returncode != 0:
    raise RuntimeError(
        f"Поиск завершился с кодом "
        f"{search_result.returncode}"
    )

Всего trial: 40
Состояния: {'COMPLETE': 25, 'PRUNED': 15}
Лучший trial: 39
Лучшая val_acc: 0.9833333333333333
Лучшие параметры: {'lr': 0.005756867015168353, 'n_layers': 3, 'hidden': 438, 'dropout': 0.050270207171060705, 'weight_decay': 1.7694297178024922e-05, 'batch_size': 128}

Сообщения Optuna:
[I 2026-11-06 01:23:34,985] Using an existing study with `study_name='lr0_search'` instead of creating a new one.
[I 2026-11-06 01:23:40,438] Trial 10 finished with value: 0.9694444444444444 and parameters: {'lr': 0.001955159293741144, 'n_layers': 3, 'hidden': 300, 'dropout': 0.09637023271546555, 'weight_decay': 1.4418961473459677e-06, 'batch_size': 128}. Best is trial 10 with value: 0.9694444444444444.
[I 2026-11-06 01:23:44,491] Trial 11 finished with value: 0.9805555555555555 and parameters: {'lr': 0.00289204272998777, 'n_layers': 3, 'hidden': 289, 'dropout': 0.09765972692147483, 'weight_decay': 1.0484348865324308e-06, 'batch_size': 128}. Best is trial 11 with value: 0.9805555555555555.
[I 

In [26]:
import subprocess
import sys

final_runs = []

for seed in range(101, 106):
    final_runs.append(
        ("final_base", "configs/base.yaml", seed)
    )
    final_runs.append(
        ("final_best", "configs/best.yaml", seed)
    )

for number, (series, config_path, seed) in enumerate(
    final_runs,
    start=1,
):
    print(
        f"\nЗапуск {number}/10: "
        f"{series}, seed={seed}"
    )

    completed = subprocess.run(
        [
            sys.executable,
            "train.py",
            "--config",
            config_path,
            "--series",
            series,
            "--seed",
            str(seed),
            "--eval-test",
            "--final-run",
        ],
        check=True,
        capture_output=True,
        text=True,
    )

    print(completed.stdout)


Запуск 1/10: final_base, seed=101
Запуск завершён: 20261106T013150Z_3f2b39f2
Каталог: runs/20261106T013150Z_3f2b39f2
Метрики: {'val_loss': 0.19473887979984283, 'training_time_seconds': 0.844611284999246, 'val_acc': 0.9555555555555556, 'test_acc': 0.9555555555555556}


Запуск 2/10: final_best, seed=101
Запуск завершён: 20261106T013201Z_8a22263e
Каталог: runs/20261106T013201Z_8a22263e
Метрики: {'val_loss': 0.1352245956659317, 'training_time_seconds': 4.262288878999243, 'val_acc': 0.9583333333333334, 'test_acc': 0.9444444444444444}


Запуск 3/10: final_base, seed=102
Запуск завершён: 20261106T013218Z_be71ab1b
Каталог: runs/20261106T013218Z_be71ab1b
Метрики: {'val_loss': 0.2016175538301468, 'training_time_seconds': 0.8362671930017314, 'val_acc': 0.95, 'test_acc': 0.9416666666666667}


Запуск 4/10: final_best, seed=102
Запуск завершён: 20261106T013229Z_70696ef9
Каталог: runs/20261106T013229Z_70696ef9
Метрики: {'val_loss': 0.12284401804208755, 'training_time_seconds': 4.500447799001449, 'va

In [27]:
import json
import statistics
from pathlib import Path

import pandas as pd
import optuna


latest_runs = {}

for run_directory in Path("runs").iterdir():
    metadata_path = run_directory / "metadata.json"
    metrics_path = run_directory / "metrics.json"

    if not metadata_path.exists() or not metrics_path.exists():
        continue

    metadata = json.loads(
        metadata_path.read_text(encoding="utf-8")
    )

    if (
        metadata["status"] != "completed"
        or metadata["series"]
        not in {"final_base", "final_best"}
    ):
        continue

    metrics = json.loads(
        metrics_path.read_text(encoding="utf-8")
    )

    key = (metadata["series"], metadata["seed"])
    candidate = {
        "directory": run_directory,
        "metadata": metadata,
        "metrics": metrics,
    }

    previous = latest_runs.get(key)

    if (
        previous is None
        or metadata["started_at"]
        > previous["metadata"]["started_at"]
    ):
        latest_runs[key] = candidate


val_deltas = []
test_deltas = []
total_fixed = 0
total_spoiled = 0

print(
    "seed | base_val | best_val | Δval "
    "| base_test | best_test | Δtest"
)

for seed in range(101, 106):
    base = latest_runs[("final_base", seed)]
    best = latest_runs[("final_best", seed)]

    base_val = base["metrics"]["val_acc"]
    best_val = best["metrics"]["val_acc"]
    base_test = base["metrics"]["test_acc"]
    best_test = best["metrics"]["test_acc"]

    delta_val = best_val - base_val
    delta_test = best_test - base_test

    val_deltas.append(delta_val)
    test_deltas.append(delta_test)

    base_predictions = pd.read_csv(
        base["directory"] / "predictions.csv"
    )
    best_predictions = pd.read_csv(
        best["directory"] / "predictions.csv"
    )

    base_test_rows = base_predictions[
        base_predictions["split"] == "test"
    ]
    best_test_rows = best_predictions[
        best_predictions["split"] == "test"
    ]

    comparison = base_test_rows.merge(
        best_test_rows,
        on=["split", "index", "y_true"],
        suffixes=("_base", "_best"),
    )

    base_correct = (
        comparison["y_pred_base"]
        == comparison["y_true"]
    )
    best_correct = (
        comparison["y_pred_best"]
        == comparison["y_true"]
    )

    total_fixed += int(
        (~base_correct & best_correct).sum()
    )
    total_spoiled += int(
        (base_correct & ~best_correct).sum()
    )

    print(
        f"{seed} | "
        f"{base_val:.4f} | {best_val:.4f} | "
        f"{delta_val:+.4f} | "
        f"{base_test:.4f} | {best_test:.4f} | "
        f"{delta_test:+.4f}"
    )


study = optuna.load_study(
    study_name="lr0_search",
    storage="sqlite:///optuna.db",
)

best_val_new_seeds = [
    latest_runs[("final_best", seed)]
    ["metrics"]["val_acc"]
    for seed in range(101, 106)
]

best_test_new_seeds = [
    latest_runs[("final_best", seed)]
    ["metrics"]["test_acc"]
    for seed in range(101, 106)
]

base_spread = (
    0.9611111111111111
    - 0.9555555555555556
)

print("\nΔval:")
print("Все значения:", val_deltas)
print("Медиана:", statistics.median(val_deltas))
print("Диапазон:", min(val_deltas), max(val_deltas))

print("\nΔtest:")
print("Все значения:", test_deltas)
print("Медиана:", statistics.median(test_deltas))
print("Диапазон:", min(test_deltas), max(test_deltas))

print("\nОшибки на test суммарно по пяти seed:")
print("Исправлено лучшей конфигурацией:", total_fixed)
print("Испорчено лучшей конфигурацией:", total_spoiled)

print("\nТри сравниваемые оценки:")
print("Лучший trial на val:", study.best_value)
print(
    "Средняя best val на новых seed:",
    statistics.mean(best_val_new_seeds),
)
print(
    "Средняя best test на новых seed:",
    statistics.mean(best_test_new_seeds),
)

print("\nРазброс базовой конфигурации:", base_spread)

stable = all(
    delta > base_spread
    for delta in test_deltas
)

print(
    "Улучшение устойчиво по критерию задания:",
    stable,
)

seed | base_val | best_val | Δval | base_test | best_test | Δtest
101 | 0.9556 | 0.9583 | +0.0028 | 0.9556 | 0.9444 | -0.0111
102 | 0.9500 | 0.9722 | +0.0222 | 0.9417 | 0.9722 | +0.0306
103 | 0.9556 | 0.9778 | +0.0222 | 0.9417 | 0.9667 | +0.0250
104 | 0.9500 | 0.9722 | +0.0222 | 0.9500 | 0.9611 | +0.0111
105 | 0.9528 | 0.9750 | +0.0222 | 0.9556 | 0.9806 | +0.0250

Δval:
Все значения: [0.002777777777777768, 0.022222222222222254, 0.022222222222222143, 0.022222222222222254, 0.022222222222222254]
Медиана: 0.022222222222222254
Диапазон: 0.002777777777777768 0.022222222222222254

Δtest:
Все значения: [-0.011111111111111183, 0.030555555555555558, 0.025000000000000022, 0.011111111111111183, 0.02499999999999991]
Медиана: 0.02499999999999991
Диапазон: -0.011111111111111183 0.030555555555555558

Ошибки на test суммарно по пяти seed:
Исправлено лучшей конфигурацией: 54
Испорчено лучшей конфигурацией: 25

Три сравниваемые оценки:
Лучший trial на val: 0.9833333333333333
Средняя best val на новых see

In [30]:
analysis_result = subprocess.run(
    [
        sys.executable,
        "analyze.py",
    ],
    capture_output=True,
    text=True,
)

print(analysis_result.stdout)

if analysis_result.returncode != 0:
    print(analysis_result.stderr)
    raise RuntimeError("Анализ завершился с ошибкой")

Эксперимент 1: {'minimum': 0.9555555555555556, 'median': 0.9583333333333334, 'maximum': 0.9611111111111111, 'range': 0.005555555555555536}
Optuna: {'total_trials': 40, 'states': {'COMPLETE': 25, 'PRUNED': 15}, 'maximum_epochs': 1200, 'actual_epochs': 833, 'saved_epochs': 367, 'saved_epoch_fraction': 0.30583333333333335, 'actual_training_seconds': 81.05109614500223, 'estimated_saved_training_seconds': 28.695394150994606, 'best_trial': 39, 'best_value': 0.9833333333333333, 'best_params': {'lr': 0.005756867015168353, 'n_layers': 3, 'hidden': 438, 'dropout': 0.050270207171060705, 'weight_decay': 1.7694297178024922e-05, 'batch_size': 128}, 'param_importance': {'dropout': 0.4186593036896728, 'lr': 0.33802605434810423, 'hidden': 0.19506248705998203, 'batch_size': 0.025421634053412915, 'n_layers': 0.01628286471830273, 'weight_decay': 0.00654765613052524}}
Финальное сравнение: {'delta_val_values': [0.002777777777777768, 0.022222222222222254, 0.022222222222222143, 0.022222222222222254, 0.0222222

In [31]:
import copy
import subprocess
import sys
from pathlib import Path

import yaml

base_config = yaml.safe_load(
    Path("configs/base.yaml").read_text(
        encoding="utf-8"
    )
)

invalid_config = copy.deepcopy(base_config)
invalid_config["train"]["lr"] = -1

invalid_path = Path("/content/invalid_config.yaml")
invalid_path.write_text(
    yaml.safe_dump(
        invalid_config,
        allow_unicode=True,
        sort_keys=True,
    ),
    encoding="utf-8",
)

result = subprocess.run(
    [
        sys.executable,
        "train.py",
        "--config",
        str(invalid_path),
        "--series",
        "validation_test",
    ],
    capture_output=True,
    text=True,
)

print("Код завершения:", result.returncode)
print(result.stderr)

assert result.returncode != 0
assert "lr должен быть положительным" in result.stderr

print("Некорректная конфигурация правильно отклонена")

Код завершения: 1
Traceback (most recent call last):
  File "/content/lr0_git/train.py", line 535, in <module>
    main()
    ~~~~^^
  File "/content/lr0_git/train.py", line 523, in main
    result = run_experiment(
        config,
    ...<2 lines>...
        final_run=args.final_run,
    )
  File "/content/lr0_git/train.py", line 280, in run_experiment
    validate_config(config)
    ~~~~~~~~~~~~~~~^^^^^^^^
  File "/content/lr0_git/train.py", line 91, in validate_config
    raise ValueError("lr должен быть положительным")
ValueError: lr должен быть положительным

Некорректная конфигурация правильно отклонена


In [32]:
from google.colab import files
from pathlib import Path
import zipfile

project_root = Path.cwd()
archive_path = Path("/content/lr0_final_artifacts.zip")

with zipfile.ZipFile(
    archive_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for target_name in [
        "runs",
        "mlflow.db",
        "mlruns",
        "optuna.db",
        "reports",
    ]:
        target = project_root / target_name

        if target.is_file():
            archive.write(
                target,
                target.relative_to(project_root),
            )

        elif target.is_dir():
            for path in target.rglob("*"):
                if path.is_file():
                    archive.write(
                        path,
                        path.relative_to(project_root),
                    )

print("Создан архив:", archive_path)
print(
    "Размер:",
    round(archive_path.stat().st_size / 1024 / 1024, 2),
    "МБ",
)

files.download(str(archive_path))

Создан архив: /content/lr0_final_artifacts.zip
Размер: 0.8 МБ


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>